# Загрузка эмбеддингов в Qdrant и примеры ANN-поиска

1. Читаем эмбеддинги из Redis (записаны ноутбуком `train_als`)
2. Создаём коллекцию в Qdrant и загружаем item-эмбеддинги
3. Примеры ANN-запросов: поиск похожих айтемов, рекомендации для пользователя, фильтрация по payload

In [ ]:
import json
import numpy as np
import pandas as pd
import redis
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue,
    Range,
)

## 1. Читаем эмбеддинги из Redis

In [ ]:
r = redis.Redis(host="localhost", port=6379, password="recsys_redis_pass", decode_responses=True)

# Item IDs и эмбеддинги
item_ids = json.loads(r.get("als:item_ids"))

pipe = r.pipeline()
for iid in item_ids:
    pipe.get(f"als:item:{iid}")
raw_embeddings = pipe.execute()

item_embeddings = {}
for iid, raw in zip(item_ids, raw_embeddings):
    if raw:
        item_embeddings[iid] = json.loads(raw)

embedding_dim = len(next(iter(item_embeddings.values())))
print(f"Items: {len(item_embeddings)}, dim: {embedding_dim}")

In [ ]:
# Загружаем метаданные айтемов для payload
items_df = pd.read_csv("../../../data/items.csv")
items_meta = items_df.set_index("item_id")[["title", "genres", "release_year", "age_rating", "content_type"]].to_dict("index")
print(f"Items metadata: {len(items_meta)}")
items_df.head(2)

## 2. Создаём коллекцию и загружаем item-эмбеддинги в Qdrant

In [ ]:
qdrant = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "items"

# Удаляем коллекцию, если существует (для идемпотентности)
if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embedding_dim,
        distance=Distance.DOT,  # ALS-эмбеддинги — dot product
    ),
)
print(f"Collection '{COLLECTION_NAME}' created, dim={embedding_dim}, distance=Dot")

In [ ]:
# Загружаем батчами по 1000 точек
BATCH_SIZE = 1000
points = []

for iid, emb in item_embeddings.items():
    meta = items_meta.get(iid, {})
    payload = {
        "item_id": iid,
        "title": meta.get("title", ""),
        "genres": meta.get("genres", ""),
        "release_year": int(meta["release_year"]) if pd.notna(meta.get("release_year")) else None,
        "age_rating": int(meta["age_rating"]) if pd.notna(meta.get("age_rating")) else None,
        "content_type": meta.get("content_type", ""),
    }
    points.append(PointStruct(id=iid, vector=emb, payload=payload))

    if len(points) >= BATCH_SIZE:
        qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
        points = []

if points:
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)

info = qdrant.get_collection(COLLECTION_NAME)
print(f"Uploaded {info.points_count} points to '{COLLECTION_NAME}'")

## 3. Примеры ANN-запросов

### 3.1 Поиск похожих айтемов (item-to-item)

In [ ]:
# Берём первый айтем и ищем 5 ближайших
query_item_id = item_ids[2]
query_vector = item_embeddings[query_item_id]

results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=6,  # +1 потому что сам айтем тоже вернётся
)

query_title = items_meta.get(query_item_id, {}).get("title", "?")
print(f"Похожие на '{query_title}' (id={query_item_id}):\n")
for point in results.points:
    if point.id == query_item_id:
        continue
    print(f"  score={point.score:.16f}  {point.payload['title']}  ({point.payload['genres']})")

### 3.2 Рекомендации для пользователя (user-to-item ANN)

In [ ]:
# Берём эмбеддинг пользователя из Redis и ищем ближайшие айтемы в Qdrant
# test_user_id = item_ids[2]  # просто для примера возьмём любой существующий user

# Найдём реального пользователя
user_keys = r.keys("als:user:*")
test_user_id = int(user_keys[8].split(":")[-1])

user_emb = json.loads(r.get(f"als:user:{test_user_id}"))
print(user_emb)
results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=user_emb,
    limit=10,
)

print(f"Top-10 рекомендаций для user_id={test_user_id}:\n")
for i, point in enumerate(results.points, 1):
    print(f"  {i}. score={point.score:.4f}  {point.payload['title']}  ({point.payload['genres']})")

### 3.3 ANN с фильтрацией по payload

Qdrant позволяет фильтровать результаты поиска по метаданным — например, только фильмы, или только контент с определённым возрастным рейтингом.

In [ ]:
# Только фильмы (content_type == "film")
results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=user_emb,
    query_filter=Filter(
        must=[
            FieldCondition(key="content_type", match=MatchValue(value="film")),
        ]
    ),
    limit=5,
)

print(f"Top-5 фильмов для user_id={test_user_id}:\n")
for i, point in enumerate(results.points, 1):
    print(f"  {i}. score={point.score:.4f}  {point.payload['title']}  ({point.payload['content_type']})")

In [ ]:
# Фильтрация по возрастному рейтингу (age_rating <= 12) — контент для детей
results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=user_emb,
    query_filter=Filter(
        must=[
            FieldCondition(key="age_rating", range=Range(lte=12)),
        ]
    ),
    limit=5,
)

print(f"Top-5 детского контента для user_id={test_user_id}:\n")
for i, point in enumerate(results.points, 1):
    p = point.payload
    print(f"  {i}. score={point.score:.4f}  {p['title']}  (age_rating={p['age_rating']})")

In [ ]:
# Комбинированный фильтр: фильмы после 2010 года с рейтингом 16+
results = qdrant.query_points(
    collection_name=COLLECTION_NAME,
    query=user_emb,
    query_filter=Filter(
        must=[
            FieldCondition(key="content_type", match=MatchValue(value="film")),
            FieldCondition(key="release_year", range=Range(gte=2010)),
            FieldCondition(key="age_rating", range=Range(lte=16)),
        ]
    ),
    limit=5,
)

print(f"Top-5 фильмов (2010+, age≤16) для user_id={test_user_id}:\n")
for i, point in enumerate(results.points, 1):
    p = point.payload
    print(f"  {i}. score={point.score:.4f}  {p['title']}  (year={p['release_year']}, age={p['age_rating']})")

### 3.4 Сравнение: brute-force vs ANN

На нашем датасете (~14K айтемов, dim=32) brute-force **будет быстрее** — numpy перемножает маленькую матрицу за доли миллисекунды (BLAS), а Qdrant платит за HTTP round-trip на каждый запрос.

ANN выигрывает на **больших данных** (100K+ векторов, высокая размерность), когда brute-force перестаёт помещаться в кэш CPU.

Но даже на малых данных Qdrant даёт преимущества, которых нет у numpy:
- **Фильтрация по payload** без полного перебора (примеры выше)
- **Персистентность** — данные на диске, не в RAM процесса
- **Отдельный сервис** — масштабируется независимо от приложения
- **Инкрементальное обновление** — добавление/удаление точек без перестройки

In [ ]:
import time

user_vec = np.array(user_emb, dtype=np.float32)
all_item_vecs = np.array([item_embeddings[iid] for iid in item_ids], dtype=np.float32)

N_ITER = 100

# Brute-force (numpy)
t0 = time.perf_counter()
for _ in range(N_ITER):
    scores = all_item_vecs @ user_vec
    top_k_bf = np.argsort(scores)[::-1][:10]
t_bf = (time.perf_counter() - t0) / N_ITER

# Qdrant ANN (включает HTTP round-trip)
t0 = time.perf_counter()
for _ in range(N_ITER):
    qdrant.query_points(collection_name=COLLECTION_NAME, query=user_emb, limit=10)
t_ann = (time.perf_counter() - t0) / N_ITER

# Qdrant ANN + фильтрация (то, чего numpy не умеет из коробки)
t0 = time.perf_counter()
for _ in range(N_ITER):
    qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=user_emb,
        query_filter=Filter(
            must=[
                FieldCondition(key="content_type", match=MatchValue(value="film")),
                FieldCondition(key="age_rating", range=Range(lte=16)),
            ]
        ),
        limit=10,
    )
t_ann_filtered = (time.perf_counter() - t0) / N_ITER

print(f"{'Метод':<35} {'Latency':>10}")
print(f"{'-'*46}")
print(f"{'Brute-force (numpy)':<35} {t_bf*1000:>8.2f} ms")
print(f"{'Qdrant ANN':<35} {t_ann*1000:>8.2f} ms")
print(f"{'Qdrant ANN + filter (film, age≤16)':<35} {t_ann_filtered*1000:>8.2f} ms")
print()
print(f"На {len(item_ids)} айтемах (dim={len(user_emb)}) numpy быстрее — это нормально.")
print(f"Overhead Qdrant ≈ {(t_ann - t_bf)*1000:.1f} ms (HTTP round-trip).")
print(f"Фильтрация добавляет всего {(t_ann_filtered - t_ann)*1000:.2f} ms — а в numpy её нужно писать руками.")